<a href="https://vigneashpandiyan.github.io/publications/Codes/" target="_blank" rel="noopener noreferrer">
  <img src="https://vigneashpandiyan.github.io/images/Link.png"
       style="max-width: 800px; width: 100%; height: auto;">
</a>

# Building Neural Net Model, Block by Block:

In [ ]:
import os
import sys
import torch
import numpy as np
import torch.nn as nn
from torchvision import transforms, datasets
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import torchvision

## Base Module Class


`torch.nn.Module` is the base class for building all neural network components in PyTorch. Modules can also contain other Modules, allowing to nest them in a tree structure.

We define a model by subclassing nn.Module, creating layers in `__init__`, and specifying how data flows through them in `forward()`.

PyTorch automatically tracks learnable parameters (e.g., weights/biases in `nn.Linear`) and registers submodules you assign as attributes.

You get useful utilities like `.parameters()` (for optimizers), `.to(device)` (move to CPU/GPU), `.train()` / `.eval()` (toggle training behavior), and `.state_dict()` (save/load weights).


Full functions:

    1. add_module - Adds a child module to the current module.
            - name (string) – name of the child module. The child module can be accessed from this module using the given name
            - parameter (Module) – child module to be added to the module.
            
    2. apply - Applies a function recursively to every submodule (as returned by .children()) as well as self.
            - fn (Module -> None) – function to be applied to each submodule
            
    3. children - Returns an iterator over immediate children modules.
    
    4. cpu - Moves all model parameters and buffers to the CPU.
    
    5. cuda - Moves all model parameters and buffers to the GPU. This also makes associated parameters and buffers different objects. So it should be called before constructing optimizer if the module will live on GPU while being optimized.
            - device (int, optional) – if specified, all parameters will be copied to that device
            
    6. double - Casts all floating point parameters and buffers to double datatype.
    
    7. eval - Sets the module in evaluation mode.
    
    8. extra_repr - Set the extra representation of the module
    
    9. float - Casts all floating point parameters and buffers to float datatype.
    
    10. forward - Defines the computation performed at every call.
    
    11. half - Casts all floating point parameters and buffers to half datatype.
    
    12. load_state_dict - Copies parameters and buffers from state_dict into this module and its descendants. If strict is True, then the keys of state_dict must exactly match the keys returned by this module’s state_dict() function.
            - state_dict (dict) – a dict containing parameters and persistent buffers.
            - strict (bool, optional) – whether to strictly enforce that the keys in state_dict match the keys returned by this module’s state_dict() function. Default: True

    13. modules() - Returns an iterator over all modules in the network.


A module’s children are the immediate submodules that were assigned as attributes inside a `nn.Module` (layers, nn.Sequential, etc.). Below we assign two `nn.Conv2D` submodules.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

'''
input -> conv2d -> relu -> conv2d -> relu -> output
'''
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.conv1 = nn.Conv2d(1, 20, 5)
        self.conv2 = nn.Conv2d(20, 20, 5)

    def forward(self, x):
       x = F.relu(self.conv1(x))
       return F.relu(self.conv2(x))

m = Model()


To correctly print the children of the model `m`, you need to call the `children()` method and then iterate over the returned iterator. Each item in the iteration will be a submodule.

In [ ]:
print('Correctly printing model children:')
for i, child in enumerate(m.children()):
    print(f'Child {i}: {child}')

Below, `dir(m)` prints a list of all attribute names you can access on model 'm':

In [ ]:
m = Model()
print(dir(m))

In [ ]:
"Printing model"
print(m)

In [ ]:
"Printing model childrens"
print(m.children)

## Sequential class

A sequential container. Modules will be added to it in the order they are passed in the constructor. Alternatively, an ordered dict of modules can also be passed in. This is the most straightforward way of building simple neural networks.

In [ ]:
# Example squential network

model = nn.Sequential(
          nn.Conv2d(1,20,5),
          nn.ReLU(),
          nn.Conv2d(20,64,5),
          nn.ReLU()
        )

print(model)


### Module List and Module Dictionaries

1. Module List - Holds submodules in a list. ModuleList can be indexed like a regular Python list, but modules it contains are properly registered, and will be visible by all Module methods.

2. Module Dict - Holds submodules in a dictionary. ModuleDict can be indexed like a regular Python dictionary, but modules it contains are properly registered, and will be visible by all Module methods.

In [ ]:
# Module list

class MyModule(nn.Module):
    def __init__(self):
        super(MyModule, self).__init__()
        self.linears = nn.ModuleList([nn.Linear(10, 10) for i in range(10)])

    def forward(self, x):
        # ModuleList can act as an iterable, or be indexed using ints
        for i, l in enumerate(self.linears):
            x = self.linears[i // 2](x) + l(x)
        return x

module_list_example = MyModule()
print(module_list_example)

In [ ]:
# Module dict

class MyModule(nn.Module):
    def __init__(self):
        super(MyModule, self).__init__()
        self.choices = nn.ModuleDict({
                'conv': nn.Conv2d(10, 10, 3),
                'pool': nn.MaxPool2d(3)
        })
        self.activations = nn.ModuleDict([
                ['lrelu', nn.LeakyReLU()],
                ['prelu', nn.PReLU()]
        ])

    def forward(self, x, choice, act):
        x = self.choices[choice](x)
        x = self.activations[act](x)
        return x

module_dict_example = MyModule()
print(module_dict_example)

Demonstrating various operations of `torch.nn.Module` in PyTorch, including defining a base model, adding submodules dynamically, applying functions recursively, moving models between devices and changing data types, setting evaluation mode, customizing model representation, saving and loading model states, counting parameters, and calculating FLOPs.

## Defining a base Model


Defining a PyTorch `nn.Module` class named `SimpleCNN` with specific convolutional, pooling, and fully connected layers, along with its `forward` method.



In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 1 input channel, 32 output channels, 3x3 kernel
        self.conv1 = nn.Conv2d(1, 32, 3)
        # 2x2 max pooling with stride 2
        self.pool = nn.MaxPool2d(2, 2)
        # 32 input channels (from conv1), 64 output channels, 3x3 kernel
        self.conv2 = nn.Conv2d(32, 64, 3)
        # Fully connected layer: 64 channels * 6x6 feature map size -> 128 output features
        self.fc1 = nn.Linear(64 * 6 * 6, 128)
        # Fully connected layer: 128 input features -> 10 output features (e.g., for 10 classes)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Apply conv1, ReLU, then pooling
        x = self.pool(F.relu(self.conv1(x)))
        # Apply conv2, ReLU, then pooling
        x = self.pool(F.relu(self.conv2(x)))
        # Flatten the output for the fully connected layers
        # x.shape[0] is the batch size, -1 infers the remaining dimension
        x = x.view(-1, 64 * 6 * 6)
        # Apply fc1, then ReLU
        x = F.relu(self.fc1(x))
        # Apply fc2 for final output
        x = self.fc2(x)
        return x

print("SimpleCNN class defined successfully.")

## Adding modules to Model


Instantiating the `SimpleCNN` model defined above, adding a new module dynamically using `add_module`, and then iterating through and printing its immediate child modules using `children()`.



In [ ]:
import torch.nn as nn

# 1. Instantiate the SimpleCNN model
model = SimpleCNN()
print("Original Model:")
print(model)

# 2. Dynamically add a new fully connected layer named 'fc3'
model.add_module('fc3', nn.Linear(10, 5))
print("\nModel after adding 'fc3':")
print(model)

# 3. Iterate through and print all immediate child modules using model.children()
print("\nChildren of the model:")
for i, child in enumerate(model.children()):
    print(f"Child {i}: {child}")

## Apply method



To demonstrate the `apply` method, we will define a custom weight initialization function, print initial weights, apply the function to the model and then print the weights again to show the effect of the initialization.



In [ ]:
import torch.nn as nn
import torch.nn.init as init

# 1. Define a custom weight initialization function
def weights_init(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        print(f"Initializing module: {m.__class__.__name__}")
        # Initialize weights using Xavier uniform distribution
        init.xavier_uniform_(m.weight.data)
        # Initialize biases to zeros if they exist
        if m.bias is not None:
            init.zeros_(m.bias.data)

# Create a new instance of the model for a clean demonstration
model_for_apply = SimpleCNN()

print("\n--- Before applying custom initialization ---")
# Optionally, print some weights before applying the function
print(f"model_for_apply.conv1.weight.data[0, 0, :, :]:\n{model_for_apply.conv1.weight.data[0, 0, :, :]}")
print(f"model_for_apply.fc1.weight.data[0, :5]:\n{model_for_apply.fc1.weight.data[0, :5]}")

# 4. Apply the weights_init function to the model
print("\n--- Applying custom initialization ---")
model_for_apply.apply(weights_init)

print("\n--- After applying custom initialization ---")
# Print the same weights after applying the function to observe the change
print(f"model_for_apply.conv1.weight.data[0, 0, :, :]:\n{model_for_apply.conv1.weight.data[0, 0, :, :]}")
print(f"model_for_apply.fc1.weight.data[0, :5]:\n{model_for_apply.fc1.weight.data[0, :5]}")


## Device and Data Type Operations


We create a new instance of the model, print its initial device and data type, then move it to CPU, CUDA (if available), and finally change its data types to double, float, and half, printing confirmations at each step.



In [ ]:
import torch
import torch.nn as nn

# 1. Instantiate a new SimpleCNN model
model_for_device_dtype = SimpleCNN()

print("--- Initial Model State ---")
# 2. Print the initial device and data type of the model's parameters
initial_device = model_for_device_dtype.conv1.weight.device
initial_dtype = model_for_device_dtype.conv1.weight.dtype
print(f"Initial device of conv1.weight: {initial_device}")
print(f"Initial data type of conv1.weight: {initial_dtype}")

print("\n--- Moving model to CPU ---")
# 3. Move the model to CPU
model_for_device_dtype.cpu()
# 4. Print the device of the model's parameters again to confirm the move to CPU
current_device_cpu = model_for_device_dtype.conv1.weight.device
print(f"Device after moving to CPU: {current_device_cpu}")

# 5. Check if CUDA is available and move to CUDA
if torch.cuda.is_available():
    print("\n--- Moving model to CUDA ---")
    model_for_device_dtype.cuda()
    current_device_cuda = model_for_device_dtype.conv1.weight.device
    print(f"Device after moving to CUDA: {current_device_cuda}")
    # 6. Move the model back to CPU if it was moved to CUDA
    print("\n--- Moving model back to CPU ---")
    model_for_device_dtype.cpu()
    current_device_back_cpu = model_for_device_dtype.conv1.weight.device
    print(f"Device after moving back to CPU: {current_device_back_cpu}")
else:
    print("\nCUDA is not available. Skipping CUDA demonstration.")

print("\n--- Changing data type to double (float64) ---")
# 7. Change the model's parameters to double precision
model_for_device_dtype.double()
current_dtype_double = model_for_device_dtype.conv1.weight.dtype
print(f"Data type after model.double(): {current_dtype_double}")

print("\n--- Changing data type to float (float32) ---")
# 8. Change the model's parameters to float precision
model_for_device_dtype.float()
current_dtype_float = model_for_device_dtype.conv1.weight.dtype
print(f"Data type after model.float(): {current_dtype_float}")

print("\n--- Changing data type to half (float16) ---")
# 9. Change the model's parameters to half precision
model_for_device_dtype.half()
current_dtype_half = model_for_device_dtype.conv1.weight.dtype
print(f"Data type after model.half(): {current_dtype_half}")


## extra_repr Method


`extra_repr()` is a method of `torch.nn.Module` we can override to customize what gets shown when we `print(model)` or `repr(model)`.

To demonstrate the `extra_repr` method, we will redefine the `SimpleCNN` class to include this method, providing a custom string representation. Then, we will instantiate and print the modified model to show the effect.



In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 1 input channel, 32 output channels, 3x3 kernel
        self.conv1 = nn.Conv2d(1, 32, 3)
        # 2x2 max pooling with stride 2
        self.pool = nn.MaxPool2d(2, 2)
        # 32 input channels (from conv1), 64 output channels, 3x3 kernel
        self.conv2 = nn.Conv2d(32, 64, 3)
        # Fully connected layer input: 64 channels * 5x5 feature map size -> 128 output features
        self.fc1 = nn.Linear(64 * 5 * 5, 128)
        # Fully connected layer: 128 input features -> 10 output features (e.g., for 10 classes)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Apply conv1, ReLU, then pooling
        x = self.pool(F.relu(self.conv1(x)))
        # Apply conv2, ReLU, then pooling
        x = self.pool(F.relu(self.conv2(x)))
        # Flatten the output for the fully connected layers
        x = x.view(-1, 64 * 5 * 5)
        # Apply fc1, then ReLU
        x = F.relu(self.fc1(x))
        # Apply fc2 for final output
        x = self.fc2(x)
        return x

    def extra_repr(self):
        # Custom string representation
        return 'Convolutional Layers: 2, Linear Layers: 2, Activation: ReLU'

print("SimpleCNN class with extra_repr defined successfully.")

# Instantiate the modified model
model_with_extra_repr = SimpleCNN()

# Print the model to observe the custom representation
print("\n--- Model with custom extra_repr ---")
print(model_with_extra_repr)

## Save and Load Model State


In PyTorch, the model state is stored in a dictionary called a `state_dict`. It contains all learnable parameters (weights/biases) and persistent buffers.
Below we demonstrate how to save the model's `state_dict` to a file using `torch.save()`. Then, instantiate a new model, and load the saved state dictionary into it using `model.load_state_dict()`.


In [ ]:
import torch
import torch.nn as nn
import os

# Ensure the SimpleCNN class is defined, using the corrected version
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.fc1 = nn.Linear(64 * 5 * 5, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

    def extra_repr(self):
        return 'Convolutional Layers: 2, Linear Layers: 2, Activation: ReLU'

# 1. Instantiate a new SimpleCNN model for saving
model_to_save = SimpleCNN()
print("Original model (model_to_save) conv1 weights (first 5 values):\n", model_to_save.conv1.weight.data.view(-1)[:5])

# 2. Define a filename for saving the model's state
model_path = 'simple_cnn_state.pth'

# 3. Save the state_dict of model_to_save to the specified file
torch.save(model_to_save.state_dict(), model_path)
print(f"\nModel state_dict saved to {model_path}")

# 4. Instantiate another new SimpleCNN model for loading
model_to_load = SimpleCNN()
print("\nNew model (model_to_load) conv1 weights BEFORE loading (first 5 values):\n", model_to_load.conv1.weight.data.view(-1)[:5])

# 5. Load the saved state_dict from the file
state_dict = torch.load(model_path)

# 6. Load the state dictionary into model_to_load
model_to_load.load_state_dict(state_dict)
print(f"\nState_dict loaded into model_to_load")

# 7. Print the weights of a layer from both models to confirm
print("\nOriginal model (model_to_save) conv1 weights (first 5 values):\n", model_to_save.conv1.weight.data.view(-1)[:5])
print("Loaded model (model_to_load) conv1 weights AFTER loading (first 5 values):\n", model_to_load.conv1.weight.data.view(-1)[:5])

# Clean up the saved file
os.remove(model_path)
print(f"\nRemoved saved model file: {model_path}")

## Counting Model Parameters

Iterating through all trainable parameters in the model (e.g., using `model.parameters()`) and calculating the total number of parameters.


In [ ]:
import torch.nn as nn

# Ensure the SimpleCNN class is defined, using the corrected version
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.fc1 = nn.Linear(64 * 5 * 5, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

    def extra_repr(self):
        return 'Convolutional Layers: 2, Linear Layers: 2, Activation: ReLU'

# 1. Instantiate a new SimpleCNN model
model_for_param_count = SimpleCNN()

# 2. Initialize a variable to store the total number of parameters
total_params = 0

print("--- Counting trainable parameters ---")
# 3. Iterate through model_for_param_count.parameters()
for name, parameter in model_for_param_count.named_parameters():
    # 4. Check if a parameter requires gradients
    if parameter.requires_grad:
        param_count = parameter.numel()
        total_params += param_count
        print(f"Layer: {name}, Parameters: {param_count}")

# 5. Print the total_params
print(f"\nTotal trainable parameters in SimpleCNN model: {total_params}")

## Calculate efficiency with FLOPs

We estimate the number of Floating Point Operations (FLOPs) for the model using a dummy input. We shall do this with a library like `torchinfo`.


In [ ]:
try:
    import torchinfo
    print("torchinfo is already installed.")
except ImportError:
    print("torchinfo not found, installing...")
    !pip install torchinfo
    import torchinfo
    print("torchinfo installed and imported successfully.")


Now that `torchinfo` is confirmed to be installed, we will define the `SimpleCNN` model again to ensure the correct architecture is used, instantiate it, create a dummy input tensor, and then use `torchinfo.summary()` to calculate and display the FLOPs.



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary

# Ensure the SimpleCNN class is defined with the correct architecture for FLOPs calculation
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 1 input channel, 32 output channels, 3x3 kernel
        self.conv1 = nn.Conv2d(1, 32, 3)
        # 2x2 max pooling with stride 2
        self.pool = nn.MaxPool2d(2, 2)
        # 32 input channels (from conv1), 64 output channels, 3x3 kernel
        self.conv2 = nn.Conv2d(32, 64, 3)
        # Corrected Fully connected layer input: 64 channels * 5x5 feature map size -> 128 output features
        self.fc1 = nn.Linear(64 * 5 * 5, 128)
        # Fully connected layer: 128 input features -> 10 output features (e.g., for 10 classes)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Apply conv1, ReLU, then pooling
        x = self.pool(F.relu(self.conv1(x)))
        # Apply conv2, ReLU, then pooling
        x = self.pool(F.relu(self.conv2(x)))
        # Flatten the output for the fully connected layers
        x = x.view(-1, 64 * 5 * 5)
        # Apply fc1, then ReLU
        x = F.relu(self.fc1(x))
        # Apply fc2 for final output
        x = self.fc2(x)
        return x

# 1. Instantiate the SimpleCNN model
model_for_flops = SimpleCNN()

# 2. Create a dummy input tensor of the appropriate shape
dummy_input_size = (1, 1, 28, 28) # Batch size 1, 1 channel, 28x28 image

print(f"\n--- Model Summary with FLOPs for input size {dummy_input_size} ---")
# 3. Use torchinfo.summary() to print a summary of the model, including its FLOPs
model_summary = summary(model_for_flops, input_size=dummy_input_size, verbose=0, device='cpu')

# 4. Print the calculated total FLOPs from the summary
# The total_flops attribute gives the total FLOPs in GigaFLOPs (10^9 FLOPs)
# We convert it to MegaFLOPs (10^6 FLOPs) for better readability if it's small.
total_flops = model_summary.total_mult_adds

if total_flops is not None:
    print(f"\nTotal FLOPs: {total_flops / 1e6:.2f} MFLOPs") # Convert to MegaFLOPs
else:
    print("FLOPs calculation not available or failed.")

print(model_summary)

## Eval mode and forward pass

`model.eval()` puts the PyTorch model into evaluation mode.

We will then create a dummy input tensor and perform a forward pass through the model using `model(input_tensor)` and print the output shape.




In [ ]:
import torch
import torch.nn as nn

# 1. Instantiate a new SimpleCNN model
model_for_eval = SimpleCNN()

print("--- Initial Model State ---")
# 2. Print the initial training status of the model
print(f"Initial training status (model_for_eval.training): {model_for_eval.training}")

print("\n--- Setting model to evaluation mode ---")
# 3. Set the model to evaluation mode
model_for_eval.eval()

# 4. Print the training status again to confirm the change
print(f"Training status after model_for_eval.eval(): {model_for_eval.training}")

print("\n--- Performing a forward pass in eval mode ---")
# 5. Create a dummy input tensor for a single 28x28 grayscale image
# (batch_size, channels, height, width)
dummy_input = torch.randn(1, 1, 28, 28)
print(f"Dummy input tensor shape: {dummy_input.shape}")

# 6. Perform a forward pass
# No_grad is often used during inference to disable gradient calculations
# for efficiency, though .eval() primarily affects dropout/batchnorm.
with torch.no_grad():
    output = model_for_eval(dummy_input)

# 7. Print the shape of the output tensor
print(f"Output tensor shape after forward pass: {output.shape}")

To go back to training behavior, we use `model.train()`.